# GTEx model building with WGCNA

💡 **Environment:** `clamp-analyses` 

In [ ]:
# WGCNA - Following Mantini et al. 2024 Methodology (https://www.nature.com/articles/s41598-024-82563-9#Sec8)

library(WGCNA)
library(here)

enableWGCNAThreads(nThreads = 2)
set.seed(123)

# Load data
gtex_data <- readRDS(here("output/gtex/df_gtex_fbm_filt.rds"))

# WGCNA expects samples x genes
datExpr <- as.data.frame(t(gtex_data))

# Pick soft threshold
powers <- 1:20
sft <- pickSoftThreshold(datExpr, powerVector = powers, networkType = "unsigned", verbose = 0)

soft_power <- sft$fitIndices$Power[which(sft$fitIndices$SFT.R.sq >= 0.9)[1]]
if (is.na(soft_power)) soft_power <- 7  # Paper's value as fallback

cat("Using soft power:", soft_power, "\n")

# Build network
net <- blockwiseModules(
  datExpr,
  power = soft_power,
  networkType = "unsigned",
  TOMType = "unsigned",
  minModuleSize = 30,
  mergeCutHeight = 0.25,
  numericLabels = TRUE,
  verbose = 3,
  maxBlockSize = ncol(datExpr),
  nThreads = 2
)

# Extract module eigengenes (remove grey/unassigned)
MEs <- net$MEs
if ("ME0" %in% colnames(MEs)) {
  MEs <- MEs[, colnames(MEs) != "ME0"]
}

# Create B matrix (modules x samples)
gtex_wgcna_B <- as.data.frame(t(MEs))
colnames(gtex_wgcna_B) <- rownames(datExpr)

cat("B matrix:", nrow(gtex_wgcna_B), "modules x", ncol(gtex_wgcna_B), "samples\n")

# Save
dir.create(here("output/gtex/wgcna"), showWarnings = FALSE, recursive = TRUE)
write.csv(gtex_wgcna_B, here("output/gtex/wgcna/gtex_wgcna_B.csv"))
saveRDS(gtex_wgcna_B, here("output/gtex/wgcna/gtex_wgcna_B.rds"))
saveRDS(net, here("output/gtex/wgcna/wgcna_network.rds"))